# Assignement 1 - SD - 09 05 2025

### 1. Installation des outils

In [1]:
import os
from datetime import datetime
from pathlib import Path

from openhexa.sdk import workspace, current_run
from openhexa.toolbox.dhis2 import DHIS2
from openhexa.toolbox.dhis2.dataframe import (
get_organisation_units,          
get_organisation_unit_groups,
get_organisation_unit_levels,
get_data_element_groups,
get_data_elements,
extract_dataset,
extract_data_element_group,
extract_events)

from openhexa.toolbox.dhis2.dataframe import get_datasets

### 2. Connection to dhis2-demo

In [2]:
from dotenv import load_dotenv
import os
# Charger ton fichier
load_dotenv("variable.env")
print("DHIS2_DEMO =", os.environ.get("DHIS2_DEMO"))
print("DHIS2_DEMO_TYPE =", os.environ.get("DHIS2_DEMO_TYPE"))

DHIS2_DEMO = dhis2
DHIS2_DEMO_TYPE = dhis2


In [4]:
con = workspace.dhis2_connection("dhis2-demo")
# sle = DHIS2(con, cache_dir="/tmp/.cache")
con

DHIS2Connection(url='https://play.dhis2.org/2.39.3.1', username='admin')

In [5]:
sle = DHIS2(con)
sle.api.session.auth = (con.username, con.password)
#sle = DHIS2(con, cache_dir="/home/hexa/workspace/.cache")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [7]:
# %conda list openhexa
con

DHIS2Connection(url='https://play.dhis2.org/2.39.3.1', username='admin')

In [ ]:
os.getcwd()


### 3. Définition des paramètres

In [ ]:
niveau_analyse = 4                                                 #Help message : Varie entre 1 et 4 pour DHIS2 demo
groupe_analyse = ["Public facilities","Private Clinic"]            #Help message : Voir liste avec le nom des groupes plus bas           
niveau_agregation = 2                                              #Help message : Varie entre 1 et 4 pour DHIS2 demo

if niveau_agregation > niveau_analyse:
    
    print("Attention, parramètres incorrects!")
else:
    print("Parramètres saisis semblent ok!")                       # Vérification supplémentaires possibles

### 4. Obtention du fichier orgunits complet

In [ ]:
df = get_organisation_units(sle)

In [ ]:
levels = get_organisation_unit_levels(sle)

In [ ]:
print(type(df))
print(len(df))
ou = df.to_pandas()
ou["level"].value_counts().sort_index()

In [ ]:
oug = get_organisation_unit_groups(sle)
oug_pd = oug.to_pandas()
oug_pd['ou_count']=oug_pd['organisation_units'].apply(len)
oug_pd.head(100)

In [ ]:
# Étape 1 : Transformer organisation_units (listes) en lignes individuelles
df_exploded = oug_pd.explode("organisation_units")
# Étape 2 : Créer un dictionnaire {id_org: [groupes]} pour savoir à quels groupes chaque ID appartient
group_mapping = df_exploded.groupby("organisation_units")["name"].apply(list).to_dict()
# Étape 3 : Ajouter une colonne par groupe dans org_units_pd
for group in oug_pd["name"]:
    # Vérifier si chaque ID est dans le groupe et remplir 1 ou 0
    ou[group] = ou["id"].apply(lambda x: 1 if group in group_mapping.get(x, []) else 0)

# Affichage du DataFrame mis à jour
ou.head(10)

In [ ]:
ou["has_geometry"] = ou["geometry"].notna()
ou["missing_geometry"] = ou["geometry"].isna()

In [ ]:
# Filtrer le DataFrame en fonction du niveau 
ou_selected = ou[(ou["level"] == niveau_analyse)]

# Filtrer le DataFrame en fonction des groupes (colonnes booléennes)
ou_selected = ou_selected[ou_selected[groupe_analyse].any(axis=1)]


In [ ]:
lvl = f"level_{niveau_agregation}_id"
result = ou_selected.groupby(lvl).agg({
    "has_geometry": "sum",  # Somme des 1 pour le nombre de lignes avec géométrie
    "missing_geometry": "sum", # Nombre de lignes sans géométrie
}).reset_index()

# Calculer le pourcentage correctement (en utilisant les deux colonnes)
result["percent_geometry"] = (result["has_geometry"] / (result["has_geometry"] + result["missing_geometry"])) * 100
result["percent_geometry"] = result["percent_geometry"].round(1)

In [ ]:
ou_reduced = ou.drop_duplicates(subset=[lvl], keep='first')[['level_1_name', 'level_2_name', 'level_3_name', lvl]]

result_with_names = result.merge(
    ou_reduced,
    on=lvl,
    how='left'
)

In [ ]:
# Obtenir la date du jour au format YYYY-MM-DD
date_today = datetime.now().strftime("%Y-%m-%d")

# Vérifier s'il existe déjà un fichier avec ce nom et ajouter un numéro séquentiel
counter = 1
file_name = f"output_{date_today}_{counter}.csv"

# Si le fichier existe déjà, incrémenter le numéro
while os.path.exists(file_name):
    counter += 1
    file_name = f"output_{date_today}_{counter}.csv"

# Enregistrer le DataFrame résultant dans ce fichier CSV
result_with_names.to_csv(file_name, index=False)

# Afficher le nom du fichier créé
print(f"Fichier sauvegardé sous : {file_name} in {os.getcwd()}")

### Liste de questions et commentaire